# Phenotype residualization

Input: round 2b's ancestry-filtered keep-list (person IDs) and a phenotype list TSV. Output: one `FID IID Y` file per phenotype x transform x covariate-set combination, matching `GRM-pairs/full_grm_bin/prep_pheno.R`'s expected format.

Protocol:
1. drop physiologically-implausible values (`plausible_min`/`plausible_max` in the phenotype list TSV, `filter_plausible_range()`) -- catches data-entry/unit errors, not just statistical outliers
2. residualize `phenotype ~ covariates` (order matters from here, see Kemper et al. 2021 Methods)
3. trim residuals > 5 SD from the mean
4. standardize surviving residuals to mean 0, var 1, within each sex

Covariates: sex-at-birth + age always included. `build_covariate_sets()` is a nested staircase, not independently-toggled combinations — `base` (age only) -> `base_pcs` (+ PCs) -> `base_pcs_zip3` (+ 3-digit zip factor) -> `base_pcs_zip3_ses` (+ SES vars from `zip3_ses_map`). Every one of those also gets crossed with raw vs. rank-inverse-normal-transformed.

## Setup

**Run this cell first in a freshly restarted kernel.** If a package (e.g. `rlang`) already got auto-loaded from the system library before this cell runs -- which can happen just from kernel startup -- `.libPaths()` can't retroactively swap it for the newer version installed here; R won't unload/replace an already-attached namespace mid-session. Symptom: `namespace 'rlang' 1.1.6 is already loaded, but >= 1.1.7 is required`. Fix is a full kernel restart, not re-running cells in the same session.

In [ ]:
required_pkgs <- c("dplyr", "tidyr", "readr", "stringr", "e1071", "bigrquery", "allofus")
missing_pkgs <- required_pkgs[!sapply(required_pkgs, requireNamespace, quietly = TRUE)]
if (length(missing_pkgs) > 0) install.packages(missing_pkgs)

library(dplyr)
library(tidyr)
library(readr)
library(stringr)
library(bigrquery)
library(allofus)
source("../../scripts/local/residualize_lib.R")   # transform, residualize, export -- shared with the fake-data test

con <- aou_connect()   # BigQuery connection to the CDR, used by pull_phenotype() below

## Inputs

- `KEEP_LIST_PATH`: round 2b output (`reverse_pca_aou.ipynb`'s AoU-fit ellipsoid filter) — person IDs passing ancestry filtering. Flip to round 2's (`round2_filter.ipynb`'s 1000G-fit ellipsoid) if that's the intended cohort instead — same convention as `query_filter_check.ipynb`'s `KEEP_LIST_PATH_ROUND2`/`KEEP_LIST_PATH_ROUND2B`.
- `PHENO_LIST_PATH`: TSV describing which phenotypes to pull (see schema below).
- `PC_PATH`: round 2b's PC covariates (`reverse_pca_aou.ipynb`'s `PC_COVARIATE_PATH` output — `IID`, `PC1`..`PC20` for every sample that passed the round 2b filter).
- `ZIP3_SES_TABLE`: AoU's `zip3_ses_map` — exact join path (likely via `observation`, not directly on `person`) still needs confirming against the real workbench, flagged below.

Phenotype list TSV schema (`PHENO_LIST_PATH`):

| column | purpose |
|---|---|
| `phenotype_name` | label used in output filenames/diagnostics |
| `source` | `measurement` / `survey` / `condition` / `fitbit` / `derived_ratio` -- picks the extraction path (only `measurement` and `derived_ratio` -- the latter for `waist_hip_ratio` specifically -- are actually implemented) |
| `concept_id` | AoU concept ID (comma-separated for multiple) |
| `value_field` | optional -- which field holds the numeric value |
| `plausible_min` / `plausible_max` | physiologically-plausible range (original units) -- values outside are dropped before any modeling, see `filter_plausible_range()` in `residualize_lib.R` |
| `notes` | free text, not used programmatically |

In [ ]:
WORKSPACE_BUCKET <- path.expand("~/workspace/Data from All of Us Controlled Tier /shared-env-pilot")
ANCESTRY_BUCKET_DIR <- file.path(WORKSPACE_BUCKET, "data", "01_ancestry_filtering")
PHENOTYPE_BUCKET_DIR <- file.path(WORKSPACE_BUCKET, "data", "02_phenotype")

# round 2b: AoU-fit ellipsoid, provisional cross-check (reverse_pca_aou.ipynb) -- see
# query_filter_check.ipynb's Inputs cell for the round 2 (1000G-fit, default) alternative
ROUND2B_OUT_DIR <- file.path(ANCESTRY_BUCKET_DIR, "reverse_pca_aou")

KEEP_LIST_PATH <- file.path(ROUND2B_OUT_DIR, "round2b_keep_ids_aou_fit_r2b_p99.9999.txt")   # round 2b output, one person_id per line
PHENO_LIST_PATH <- "../../docs/phenotype_list.tsv"  # starter anthropometric/metabolic panel -- see its header for concept_id confirmation status
PC_PATH <- file.path(ROUND2B_OUT_DIR, "round2b_pc_covariates_p99.9999.txt")   # round 2b's PC1-PC20 for its retained set (reverse_pca_aou.ipynb's PC_COVARIATE_PATH output)
OUT_DIR <- file.path(PHENOTYPE_BUCKET_DIR, "residualized")          # where the FID/IID/Y files get written
MODELING_TABLE_DIR <- file.path(PHENOTYPE_BUCKET_DIR, "modeling_tables")   # one neat TSV per phenotype (prepare_modeling_tables()'s
                                       # output) -- lives in the workspace bucket (per root README's bucket
                                       # layout), not local disk: this is what lets the residualization
                                       # procedure itself get retuned later without re-pulling from BigQuery
REFERENCE_DATE <- "2024-01-01"   # arbitrary fixed date -- adjust to match the CDR version's actual data cutoff, same convention as query_filter_check.ipynb
RAW_PHENO_CACHE_DIR <- file.path(PHENOTYPE_BUCKET_DIR, "raw_pheno_cache")  # raw pull_phenotype() output per phenotype, cached so repeated
                                       # runs while iterating on residualize_lib.R don't re-hit BigQuery

dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(MODELING_TABLE_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(RAW_PHENO_CACHE_DIR, recursive = TRUE, showWarnings = FALSE)

## Read the ID list and phenotype list

`keep_ids`: everyone downstream gets restricted to this set before anything else -- residualization, transforms, and diagnostics are all computed within this cohort only, not the full AoU population.

In [ ]:
keep_ids_raw <- read_lines(KEEP_LIST_PATH)
# round 2b's keep-list is plink-style: either one ID per line, or "FID IID"
# space-separated -- take the last whitespace-separated field either way
keep_ids <- str_trim(keep_ids_raw) %>% str_split(" +") %>% sapply(function(x) tail(x, 1))

pheno_list_all <- read_tsv(PHENO_LIST_PATH, col_types = cols(.default = "c"))
stopifnot(all(c("phenotype_name", "source", "concept_id", "plausible_min", "plausible_max") %in% names(pheno_list_all)))

# UNCONFIRMED concept_ids (currently the 3 lipid rows -- see phenotype_list.tsv's
# notes column) aren't real AoU concept IDs and can't be queried; excluded here
# rather than left to fail deep inside pull_phenotype() with an opaque BigQuery
# error. Confirm the concept_id against the AoU Data Browser and edit
# phenotype_list.tsv to include those phenotypes in a run.
unconfirmed <- pheno_list_all$phenotype_name[pheno_list_all$concept_id == "UNCONFIRMED"]
if (length(unconfirmed) > 0) {
  message("Skipping phenotypes with UNCONFIRMED concept_id: ", paste(unconfirmed, collapse = ", "))
}
pheno_list <- pheno_list_all %>% filter(concept_id != "UNCONFIRMED")
pheno_list

## Pull phenotype + covariate table

`pull_phenotype()` handles `source == "measurement"` via `allofus::aou_sql()` (most recent value per person, age computed from a single fixed `REFERENCE_DATE` rather than per-measurement). Sex is `person.sex_at_birth_concept_id` (a direct AoU-specific column -- distinct from `person.gender_concept_id`, which is gender identity). `pull_covariates()`'s zip3/SES join uses `zip3_ses_map`, keyed off a masked zip-code `observation` row (AoU privacy-protects exact zip, only the 3-digit prefix survives, marked with a `*` in `value_as_string`). `survey`/`condition`/`fitbit`/`derived_ratio` phenotype sources aren't wired up here -- `pull_phenotype()` throws a clear error on any of them rather than silently doing nothing, and `prepare_modeling_tables()` catches that per-phenotype and skips it (see below) rather than crashing the whole run. `waist_hip_ratio` (`derived_ratio`) is the one exception with an actual implementation, built as its own cell further down instead of inside `pull_phenotype()`, since it doesn't fit that function's one-concept_id-in, one-value-out shape.

Everything below this cell (transform, covariate-set configs, residualization, export, diagnostics) doesn't depend on these specifics and is fully generic once `pull_phenotype()` returns a table shaped `person_id, phenotype, age, sex_at_birth`.

In [ ]:
pull_phenotype <- function(row, keep_ids) {
  cache_path <- file.path(RAW_PHENO_CACHE_DIR, paste0(row$phenotype_name, ".tsv"))
  if (file.exists(cache_path)) {
    return(read_tsv(cache_path, col_types = cols(person_id = "c", phenotype = "d", age = "d", sex_at_birth = "c")))
  }

  if (row$source != "measurement") {
    stop(sprintf(
      "pull_phenotype(): source '%s' not implemented -- only 'measurement' is wired up so far",
      row$source
    ))
  }

  if (!grepl("^[0-9]+(,[0-9]+)*$", row$concept_id)) {
    stop(sprintf(
      "pull_phenotype(): '%s' has concept_id '%s', not a valid comma-separated numeric AoU concept ID -- confirm it against the AoU Data Browser and fix docs/phenotype_list.tsv before running this phenotype",
      row$phenotype_name, row$concept_id
    ))
  }

  # DATE_DIFF returns INT64; bigrquery collects bare INT64 columns as bit64::integer64,
  # which lm() silently mis-coerces into a degenerate fit rather than erroring -- cast to
  # FLOAT64 in the query so age collects as a plain double
  query <- sprintf("
    WITH demographics AS (
      SELECT
        person_id,
        CAST(DATE_DIFF(DATE '%s', DATE(birth_datetime), YEAR) AS FLOAT64) AS age,
        CASE
          WHEN sex_at_birth_concept_id = 45878463 THEN 'Female'
          WHEN sex_at_birth_concept_id = 45880669 THEN 'Male'
          ELSE 'Other'
        END AS sex_at_birth
      FROM {CDR}.person
    ),
    measurements AS (
      SELECT
        person_id,
        value_as_number AS phenotype,
        ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY measurement_date DESC) AS rn
      FROM {CDR}.measurement
      WHERE measurement_concept_id IN (%s)
        AND value_as_number IS NOT NULL
    )
    SELECT d.person_id, m.phenotype, d.age, d.sex_at_birth
    FROM demographics d
    INNER JOIN measurements m ON d.person_id = m.person_id
    WHERE m.rn = 1
  ", REFERENCE_DATE, row$concept_id)

  pheno_df <- aou_sql(query) %>%
    collect() %>%   # pull out of BigQuery before any local joins/filters -- a
                     # lazy remote tbl can't be left_join()'d against a local one
    mutate(person_id = as.character(person_id)) %>%
    filter(person_id %in% keep_ids)

  write_tsv(pheno_df, cache_path)   # cache the raw pull -- avoids re-hitting BigQuery on every
                                     # downstream iteration; delete the file to force a refresh
  pheno_df
}

pull_covariates <- function(keep_ids) {
  pcs <- read_table(PC_PATH, col_types = cols(.default = "d", IID = "c")) %>%
    rename(person_id = IID) %>%
    filter(person_id %in% keep_ids)

  # zip3 comes from a masked (privacy-protected) zip-code observation --
  # AoU marks these rows with a "*" in value_as_string; the first 3 characters
  # are the 3-digit zip prefix, joined against the CDR-provided zip3_ses_map.
  # median_income/poverty/deprivation_index cast to FLOAT64 for the same reason
  # as age in pull_phenotype() -- an unqualified INT64 column collects as
  # bit64::integer64, which lm() silently mis-coerces rather than erroring
  zip3_ses_query <- "
    SELECT
      o.person_id,
      z.zip3,
      CAST(z.median_income AS FLOAT64) AS median_income,
      CAST(z.fraction_poverty AS FLOAT64) AS poverty,
      CAST(z.deprivation_index AS FLOAT64) AS deprivation_index
    FROM (
      SELECT person_id, CAST(SUBSTR(value_as_string, 1, 3) AS INT64) AS zip3
      FROM {CDR}.observation
      WHERE STRPOS(value_as_string, '*') > 0
    ) o
    INNER JOIN {CDR}.zip3_ses_map z ON o.zip3 = z.zip3
  "
  zip3_ses <- aou_sql(zip3_ses_query) %>%
    collect() %>%
    mutate(person_id = as.character(person_id)) %>%
    filter(person_id %in% keep_ids)

  zip3 <- zip3_ses %>% transmute(person_id, zip3 = as.character(zip3))
  ses <- zip3_ses %>% select(person_id, median_income, poverty, deprivation_index)

  list(pcs = pcs, zip3 = zip3, ses = ses)
}

## Optional skew-reducing transform

Rank-based inverse-normal transform (`inverse_normal_transform()` in `residualize_lib.R`) -- chosen over log/Box-Cox since it doesn't assume a particular skew direction or require positive values, so it works uniformly across an arbitrary phenotype list without per-phenotype tuning. Both `raw` and `invnorm` variants get carried through the rest of the pipeline; skewness before/after is reported so it's clear whether the transform actually helped for each phenotype, rather than applying it blindly. See `02_phenotype/notebooks/local/test_residualize_fake_data.ipynb` for a worked example on synthetic data showing this diagnostic actually catching a real skew reduction.

## Covariate-set configs

`build_covariate_sets()` (in `residualize_lib.R`) returns a named list of formula RHS vectors. `sex_at_birth` is handled separately (it's the stratification variable for step 3, not a residualization covariate -- see `residualize_phenotype()` in the lib), so it's not in these formulas.

In [ ]:
pc_cols <- paste0("PC", 1:5)   # top 5 only -- beyond that isn't considered informative for this cohort
covariate_sets <- build_covariate_sets(pc_cols)
names(covariate_sets)

## Prepare modeling tables

`prepare_modeling_tables()` (in `residualize_lib.R`) is the only cell below
that hits BigQuery -- pulls each phenotype, applies the plausible-range
filter, joins PCs/zip3/SES, adds the invnorm variant, and writes one neat
TSV per phenotype to `MODELING_TABLE_DIR` (`<phenotype_name>.tsv`:
`person_id, phenotype, phenotype__invnorm, age, sex_at_birth`, plus every
covariate column). Re-run this cell only when the phenotype list,
keep-list, or covariate pulls themselves change.

If `pull_phenotype()` throws for a given row (wrong/unimplemented `source`, a `concept_id` that doesn't actually exist in this CDR version, a transient BigQuery error), that phenotype is skipped -- `message()`-printed, recorded in `range_summary_table` below with `status == "skipped: pull_phenotype() failed"` -- rather than taking down every other phenotype's pull too. `pull_covariates()` is deliberately *not* given the same treatment: it runs once per phenotype but is the same call every time, so a failure there is systemic, not phenotype-specific, and should stop the run loudly.

In [ ]:
prep <- prepare_modeling_tables(
  pheno_list, keep_ids, pull_phenotype, pull_covariates, MODELING_TABLE_DIR
)
prep$range_summary_table  # per phenotype: N excluded by the plausible-range filter, before any modeling

## Derived phenotype: waist/hip ratio

`waist_hip_ratio` (`source == "derived_ratio"` in `phenotype_list.tsv`)
isn't a single pulled measurement -- `pull_phenotype()` throws on it
immediately (unimplemented source), which `prepare_modeling_tables()`'s
per-phenotype error handling turns into a skip, not a crash. This cell
builds its modeling table directly instead, by combining the
`waist_circumference`/`hip_circumference` tables the "Prepare modeling
tables" cell above already wrote (no new BigQuery calls) -- so it must run
after that cell, before "Run residualization" below.

**Caveat:** `waist`/`hip` are each independently "most recent value as of
`REFERENCE_DATE`" (same as every other phenotype here), joined on
`person_id` -- not matched to the same visit/date. In practice AoU's
Physical Measurements module tends to collect both together, so this is
usually fine, but if the ratio's distribution looks off, mismatched-visit
pairing is the first thing to check.

In [ ]:
waist_path <- file.path(MODELING_TABLE_DIR, "waist_circumference.tsv")
hip_path <- file.path(MODELING_TABLE_DIR, "hip_circumference.tsv")

if (file.exists(waist_path) && file.exists(hip_path)) {
  waist <- read_tsv(waist_path, show_col_types = FALSE) %>% select(person_id, waist = phenotype)
  hip <- read_tsv(hip_path, show_col_types = FALSE) %>% select(person_id, hip = phenotype, age, sex_at_birth)

  ratio_row <- pheno_list_all %>% filter(phenotype_name == "waist_hip_ratio")
  stopifnot(nrow(ratio_row) == 1)

  whr_df <- waist %>%
    inner_join(hip, by = "person_id") %>%
    mutate(phenotype = waist / hip) %>%
    select(person_id, phenotype, age, sex_at_birth)

  range_result <- filter_plausible_range(
    whr_df, "phenotype", as.numeric(ratio_row$plausible_min), as.numeric(ratio_row$plausible_max)
  )
  whr_df <- range_result$data

  covars <- pull_covariates(keep_ids)
  whr_df <- whr_df %>%
    left_join(covars$pcs, by = "person_id") %>%
    left_join(covars$zip3, by = "person_id") %>%
    left_join(covars$ses, by = "person_id") %>%
    add_transformed_variant("phenotype")

  write_tsv(whr_df, file.path(MODELING_TABLE_DIR, "waist_hip_ratio.tsv"))
  message(sprintf(
    "waist_hip_ratio: %d excluded as implausible, %d written to %s",
    range_result$n_excluded, nrow(whr_df), file.path(MODELING_TABLE_DIR, "waist_hip_ratio.tsv")
  ))
} else {
  message(
    "Skipping waist_hip_ratio: waist_circumference.tsv and/or hip_circumference.tsv not found in ",
    "MODELING_TABLE_DIR -- run 'Prepare modeling tables' first, and confirm both concept_ids in ",
    "phenotype_list.tsv if either was skipped as UNCONFIRMED or failed to pull"
  )
}

## Run residualization

`run_residualization_from_tables()` reads `MODELING_TABLE_DIR`'s TSVs back
in and runs the full cross product of {phenotype} x {raw, invnorm} x
{covariate-set}, all exported -- no curation, every combination in
`pheno_list` x `covariate_sets` gets written out as an `FID IID Y` file
matching `GRM-pairs/full_grm_bin/prep_pheno.R`'s expected format. No
BigQuery access needed -- this is the cell to re-run (on its own, without
re-running "Prepare modeling tables" above) when retuning
`covariate_sets`, `outlier_sd`, or which phenotypes to residualize.

A phenotype with no table in `MODELING_TABLE_DIR` (skipped above, or never prepared) is likewise skipped here rather than erroring on a missing file -- `message()`-printed, recorded in `combo_summary_table` with `status == "skipped: no modeling table found"`.

In [ ]:
result <- run_residualization_from_tables(pheno_list, MODELING_TABLE_DIR, covariate_sets, OUT_DIR)

## Diagnostics summary

In [ ]:
result$skew_summary_table   # per phenotype: skewness before/after the invnorm transform

In [ ]:
result$combo_summary_table  # per phenotype x variant x covariate-set: N retained, R^2